In [2]:
import pandas as pd
import numpy as np

from scipy.stats import t
from scipy.stats import ttest_1samp

In [15]:

file_path = '/kaggle/input/datasets/fireflyqmf313/f-kan-results-plot/Real_0226/Results_Milk.xlsx'

output_file = 'Results_Milk_NIT.xlsx'

########################################################
# Parameters
########################################################

FKAN_name = 'F-KAN'

DELTA = 0.05

metrics = [
    'Test_MSE',
    'Test_MAE',
    'Test_R2'
]

########################################################
# Read data
########################################################

xls = pd.ExcelFile(file_path)
sheet_names = xls.sheet_names

all_data = {}

for model in sheet_names:

    df = pd.read_excel(
        file_path,
        sheet_name=model
    )

    temp = {}

    for metric in metrics:

        values = (
            df[metric]
            .iloc[2:102]
            .astype(float)
            .reset_index(drop=True)
        )

        temp[metric] = values

    all_data[model] = temp


########################################################
# Non-inferiority test
########################################################

with pd.ExcelWriter(
        output_file,
        engine='openpyxl'
) as writer:

    for metric in metrics:

        fkan = all_data[FKAN_name][metric]

        results = []

        for model in sheet_names:

            if model == FKAN_name:
                continue

            x = all_data[model][metric]

            ################################################
            # Difference
            ################################################

            diff = fkan - x

            n = len(diff)

            mean_diff = np.mean(diff)

            sd = np.std(diff, ddof=1)

            se = sd / np.sqrt(n)

            ################################################
            # 95% CI
            ################################################

            tcrit = t.ppf(0.975, df=n-1)

            ci_lower = mean_diff - tcrit * se
            ci_upper = mean_diff + tcrit * se

            ################################################
            # Non-inferiority test
            ################################################

            if metric == 'Test_R2':

                # H0: mean_diff <= -DELTA
                # H1: mean_diff > -DELTA

                shifted = diff + DELTA

                stat, p_two = ttest_1samp(
                    shifted,
                    popmean=0
                )

                p_one = p_two / 2

                if stat < 0:
                    p_one = 1 - p_one

                noninferior = (
                    ci_lower > -DELTA
                )

            else:

                # MSE / MAE
                # H0: mean_diff >= DELTA
                # H1: mean_diff < DELTA

                shifted = diff - DELTA

                stat, p_two = ttest_1samp(
                    shifted,
                    popmean=0
                )

                p_one = p_two / 2

                if stat > 0:
                    p_one = 1 - p_one

                noninferior = (
                    ci_upper < DELTA
                )

            ################################################
            # Direction
            ################################################

            fkan_mean = np.mean(fkan)
            x_mean = np.mean(x)

            if metric == 'Test_R2':

                if fkan_mean > x_mean:
                    direction = 'F-KAN higher'

                elif fkan_mean < x_mean:
                    direction = 'F-KAN lower'

                else:
                    direction = 'Equal'

            else:

                if fkan_mean < x_mean:
                    direction = 'F-KAN lower'

                elif fkan_mean > x_mean:
                    direction = 'F-KAN higher'

                else:
                    direction = 'Equal'

            ################################################

            results.append([

                f'F-KAN vs {model}',

                direction,

                mean_diff,

                ci_lower,

                ci_upper,

                DELTA,

                stat,

                p_one,

                noninferior

            ])

        ####################################################
        # Save sheet
        ####################################################

        result_df = pd.DataFrame(

            results,

            columns=[

                'Comparison',

                'Direction',

                'Mean_Difference',

                'CI_Lower_95%',

                'CI_Upper_95%',

                'Delta',

                't_statistic',

                'p_value',

                'Noninferior'

            ]

        )

        result_df.to_excel(
            writer,
            sheet_name=metric,
            index=False
        )

print('\nDone.')
print(output_file)


Done.
Results_Milk_NIT.xlsx
